<a href="https://colab.research.google.com/github/liminalvoid/nlp/blob/main/sem_6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Семинар 6. Токенайзеры, размеры тензеров и attention

## Установка и импорт библиотек, начальная настройка

In [ ]:
%pip install -q transformers sentencepiece

import random

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import transformers

from transformers import (
    AutoTokenizer,
    AutoConfig,
    AutoModel,
    logging,
)


SEED = 52

torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

logging.set_verbosity_error()

## Сравнение токенайзеров

В качестве дополнительного токенайзера использован токенайзер из модели [Qwen/Qwen2.5-7B](https://huggingface.co/Qwen/Qwen2.5-7B).

Число токенов зависит от словаря и алгоритма обучения. BPE-токенайзер строит словарь, начиная с отдельных символов и постепенно склеивая самые частотные пары. Какие пары склеятся — зависит от обучающего корпуса. Если в корпусе было много русского текста, частые слоги вроде «ого», «тион», «ский» станут отдельными токенами. Если русского было мало, слово будет разбито посимвольно. Отсюда разница: BERT (обучен в основном на английском) разобьёт «французских» на 5+ токенов, а Qwen (большой мультиязычный корпус) — на 1–2.

Код имеет совершенно другое распределение символов: много скобок, отступов, camelCase, snake_case, спецсимволов, поэтому он режется иначе. Современные токенайзеры (GPT-4, Qwen, LLaMA 3) специально включают код в обучающий корпус, поэтому у них в словаре есть целые паттерны вроде `def` , `    ` (4 пробела), `self`, `return`. Кроме того, многие токенайзеры применяют регулярные выражения перед токенизацией, которые по-разному разделяют буквы, цифры и пунктуацию — это напрямую влияет на то, где пройдут границы токенов в коде и в прозе.

BPE жадно склеивает частые пары. Если при обучении токенайзера русский текст составлял 0.1% корпуса, русские биграммы просто не наберут достаточно частоты, чтобы склеиться в длинные токены — и текст будет разбит почти посимвольно. В multilingual-моделях русского больше, поэтому появляются токены-слоги и даже целые слова. Результат: меньше токенов на то же предложение, а значит больше текста влезает в контекстное окно и инференс дешевле.

Byte-level BPE (GPT-2, LLaMA, Qwen) работает не с Unicode-символами, а с сырыми байтами. Кириллическая буква в UTF-8 — это 2 байта, китайский иероглиф — 3. Если пара байтов недостаточно частотна, чтобы склеиться, вы увидите в выводе отдельные байты, которые выглядят как мусор (как в случае с GPT-2 и Qwen): Ġ, Ð, µ и т.п. Это не ошибка — это просто отображение отдельных байтов UTF-8 через маппинг в печатные символы. По мере того как токенайзер обучается на большем количестве русского текста, эти байты склеиваются в нормальные слоги и слова.

In [ ]:
def inspect_tokenizer(tok_name: str, text: str, spacing_width: int = 16):
    tok = AutoTokenizer.from_pretrained(tok_name)
    encoded = tok(text, return_tensors="pt")
    tokens = tok.convert_ids_to_tokens(encoded["input_ids"][0])

    print(f"{'Text:':<{spacing_width}} {repr(text)}")
    print(f"{'Tokens:':<{spacing_width}} {str(tokens)}")
    print(f"{'Tokens count:':<{spacing_width}} {len(tokens)}")
    print(f"{'Vocab size:':<{spacing_width}} {tok.vocab_size}")
    print(f"{'Full vocab size:':<{spacing_width}} {len(tok.get_vocab())}\n")


texts = {
    "russian": "Съешь ещё этих мягких французских булок, да выпей чаю.",
    "code": "def print_str(string: str):\n    print(string)\n",
}
tokenizer_names = {
    "bert-base-uncased": "BERT WordPiece",
    "gpt2": "GPT-2 byte-level BPE",
    "google/mt5-small": "mT5 SentencePiece",
    "xlm-roberta-base": "XLM-R SentencePiece",
    "Qwen/Qwen2.5-7B": "Qwen SentencePiece",
}

for tok_name in tokenizer_names:
    print(f"=== {tok_name} ===")

    for (lang, text), i in zip(texts.items(), range(len(texts))):
        print(f"{i + 1}) {lang}")
        inspect_tokenizer(tok_name, text)

Ниже представлена табица числа токеном для разных текстов и токенайзеров.

In [ ]:
SPACING = 24

print(f"{'Tokenizer':<{SPACING}}", end="")

for lang in texts:
    print(f"┃ {lang:<{SPACING}}", end="")

print()

line = "━"
print(line * SPACING, end="", sep="")

for _ in range(len(texts)):
    print("╋", line * (SPACING + 1), end="", sep="")

print()

for tok_name, tok_description in tokenizer_names.items():
    tok = AutoTokenizer.from_pretrained(tok_name)
    print(f"{tok_name:<{SPACING}}", end="")

    for lang, text in texts.items():
        n = len(tok(text)["input_ids"])
        print(f"┃ {n:<{SPACING}}", end="")

    print()

## Расчет размеров

Дано: $B = 2$, $T = 7$, $d_{model} = 768$ и $h = 12$.

Найти: $Q$, $K$, $V$ и $QK^T$.

Решение:

$$
    d_k = \frac{d_{model}}{h} = \frac{768}{12} = 64
$$

$$
    Q, K, V: (B, h, T, d_k) \iff (2, 12, 7, 64)
$$

$$
    QK^T: (B, h, T, T) \iff (2, 12, 7, 7)
$$

## Поиск числа слоев, голов и размерности модели

In [ ]:
def extract_basic_params(cfg):
    hidden = getattr(cfg, "hidden_size", None)
    if hidden is None:
        hidden = getattr(cfg, "d_model", None)

    heads = getattr(cfg, "num_attention_heads", None)
    if heads is None:
        heads = getattr(cfg, "num_heads", None)

    layers = getattr(cfg, "num_hidden_layers", None)
    if layers is None:
        layers = getattr(cfg, "num_layers", None)
    if layers is None:
        layers = getattr(cfg, "encoder_layers", None)

    return hidden, heads, layers


model_name = "Qwen/Qwen2.5-7B"
cfg = AutoConfig.from_pretrained(model_name)
hidden, heads, layers = extract_basic_params(cfg)
print(f"{model_name:30s} hidden={hidden}, heads={heads}, layers={layers}")

Механизм внимания данной модели имеет следующие особенности:

- Grouped Query Attention (GQA) с агрессивным соотношением  
Модель использует 28 голов запросов (`"num_attention_heads": 28`), но всего 4 головы ключей-значений (`"num_key_value_heads": 4`). Это соотношение 7:1 – каждая KV-голова разделяется между 7 головами запросов. Это довольно агрессивно по сравнению, например, с Llama 3 8B, где используется соотношение 4:1 (32Q / 8KV). Выигрыш — значительное сокращение памяти KV-кэша при инференсе, что особенно важно при контекстном окне в 128K токенов.
- QKV bias  
Архитектура включает смещения (bias) в проекциях QKV, что нетипично — большинство современных моделей (Llama, Mistral) полностью отказываются от смещений в проекциях внимания. Qwen сохраняет их начиная с Qwen1, и это одно из отличительных архитектурных решений семейства ([строка 200-203](https://github.com/huggingface/transformers/blob/a553395766001116a719c82870171f8d6b458c98/src/transformers/models/qwen2/modeling_qwen2.py#L200-L203)).
- Полное внимание на всех слоях  
Несмотря на то что архитектура Qwen2 поддерживает комбинацию скользящего оконного внимания и полного внимания, Qwen2.5-7B не использует это: `use_sliding_window` равен `False`, а `max_window_layers` совпадает с `num_hidden_layers` (оба равны 28). Каждый слой обрабатывает всю последовательность целиком. Поддержка длинного контекста до 128K токенов обеспечивается за счёт RoPE с высокой базовой частотой (`"rope_theta": 1000000.0`), а не за счёт оконных механизмов.

In [ ]:
cfg

## Поиск `W_K` и описание наиболее интересного паттерна головы внимания

Поиск `W_K`.

In [ ]:
model = AutoModel.from_pretrained(model_name, attn_implementation="eager")

for name, param in model.named_parameters():
    name = name.lower()

    if ("key" in name) or ("k_proj" in name) or ("c_attn" in name):
        print(name, tuple(param.shape))

На 28 головах последнего слоя видно несколько характерных паттернов внимания:

Диагональные головы (например, 4, 5, 6, 12, 13, 19, 20, 25, 26) обращают внимание почти исключительно на текущий или непосредственно предшествующий токен — ярко выраженная диагональная линия. Это «локальные» или «копирующие» головы, которые, вероятно, отвечают за сохранение позиционной информации и обработку ближних зависимостей.

Головы с вертикальными полосами (например, 3, 10, 17, 24). На них видны яркие вертикальные столбцы — это значит, что каждый токен сильно «смотрит» на одни и те же несколько позиций, как правило на первый токен или знаки пунктуации. Это так называемые «sink»-головы. Паттерн повторяется примерно каждые 7 голов (3, 10, 17, 24) — и это интересно, потому что 7 — это именно размер GQA-группы (28 Q-голов / 4 KV-головы = 7). Каждая группа из 7 голов запросов делит одну KV-голову, и похоже, что в каждой группе ровно одна голова берёт на себя роль «sink».

Головы с широким вниманием (например, 0, 7, 8). Здесь виден более размытый треугольный паттерн — нижний левый треугольник «подсвечен», то есть поздние токены обращают внимание на множество более ранних. Скорее всего, именно эти головы выполняют основную «семантическую» работу — агрегируют значение по всему контексту.

Почти пустые головы (например, 15, 16, 18, 21, 22, 23). Они практически полностью тёмные, с едва заметной активностью. На последнем слое многие головы уже «отработали» на более ранних слоях и вносят минимальный вклад — либо их функция слишком тонкая, чтобы проявиться на линейной цветовой шкале.

Самое любопытное наблюдение — периодичность sink-паттерна с шагом 7, которая точно совпадает с группировкой GQA. Это говорит о том, что внутри каждой KV-группы сформировалось специализированное разделение ролей между головами.

In [ ]:
model.eval()

text = "Съешь ещё этих мягких французских булок, да выпей чаю."
tok = AutoTokenizer.from_pretrained(model_name)
batch = tok(text, return_tensors="pt")

with torch.no_grad():
    out = model(**batch, output_attentions=True, return_dict=True)

    layer = -1
    num_heads = out.attentions[layer].shape[1]
    tokens = tok.convert_ids_to_tokens(batch['input_ids'][0])

    num_heads = out.attentions[layer].shape[1]
    rows = 4
    cols = (num_heads + 1) // rows

    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 5 * rows), constrained_layout=True)
    axes = axes.flatten()

    for h in range(num_heads):
        ax = axes[h]
        mat = out.attentions[layer][0, h].cpu().float().numpy()
        ax.imshow(mat, aspect='auto')
        ax.set_title(f'head {h}')
        ax.set_xticks(range(len(tokens)))
        ax.set_xticklabels(tokens, rotation=90, fontsize=7)
        ax.set_yticks(range(len(tokens)))
        ax.set_yticklabels(tokens, fontsize=7)

    # hide any unused subplots if num_heads is odd
    for h in range(num_heads, len(axes)):
        axes[h].set_visible(False)

    plt.show()
